# DATA 5310 HW 8
## Badamgarav Battushig
### Our 3rd Dataset: World Bank Country Indicators

## About this dataset

Source: World Bank (2026). *World Development Indicators*. Retrieved November 16, 2026, from https://data.worldbank.org

The World Bank publishes yearly statistics for almost every country in the world. We use three of them:

| Indicator | Code | Source page |
|---|---|---|
| Population, total | `SP.POP.TOTL` | https://data.worldbank.org/indicator/SP.POP.TOTL |
| GDP per capita (current US$) | `NY.GDP.PCAP.CD` | https://data.worldbank.org/indicator/NY.GDP.PCAP.CD |
| Individuals using the Internet (% of population) | `IT.NET.USER.ZS` | https://data.worldbank.org/indicator/IT.NET.USER.ZS |

The files were downloaded from the pages above and saved in our group's GitHub repository: https://github.com/Badamgarv-Battushig/Data-Visualization-Group-Project/tree/main/data

Why we need it: our other two datasets (EuRepoC and Ransomware.live) only count cyber attacks. A large country like the United States will naturally have more attacks than a small one, simply because it has more people, companies, and computers. This dataset tells us how many people live in each country, how wealthy it is, and how many people use the internet. With it we can make fair comparisons, such as **attacks per million people**, and check whether richer or more connected countries are targeted more often.

In [1]:
# install country_converter (only needed once)
!pip install -q country_converter

In [2]:
import pandas as pd
import altair as alt
import country_converter as coco

## Data Set

In [3]:
# GitHub location of our data files
github_url = 'https://raw.githubusercontent.com/Badamgarv-Battushig/Data-Visualization-Group-Project/main/data/'

# the 3 indicator files start with 4 rows of notes, so we skip them
population = pd.read_csv(github_url + 'wb_population.csv', skiprows=4)
gdp = pd.read_csv(github_url + 'wb_gdp_per_capita.csv', skiprows=4)
internet = pd.read_csv(github_url + 'wb_internet_users.csv', skiprows=4)

# country metadata (region and income group for each country)
metadata = pd.read_csv(github_url + 'wb_country_metadata.csv')

population.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"Population, total",SP.POP.TOTL,54922.0,55578.0,56320.0,57002.0,57619.0,58190.0,...,108735.0,108908.0,109203.0,108587.0,107700.0,107310.0,107359.0,107995.0,108785.0,NaN
1,Africa Eastern and Southern,AFE,"Population, total",SP.POP.TOTL,130075728.0,133534923.0,137171659.0,140945536.0,144904094.0,149033472.0,...,640058741.0,657801085.0,675950189.0,694446100.0,713090928.0,731821393.0,750491370.0,769280888.0,788844284.0,NaN
2,Afghanistan,AFG,"Population, total",SP.POP.TOTL,9035043.0,9214083.0,9404406.0,9604487.0,9814318.0,10036008.0,...,35688935.0,36743039.0,37856121.0,39068979.0,40000412.0,40578842.0,41454761.0,42647492.0,43844111.0,NaN
3,Africa Western and Central,AFW,"Population, total",SP.POP.TOTL,97630925.0,99706674.0,101854756.0,104089175.0,106384410.0,108754449.0,...,440150152.0,451395343.0,462522286.0,473687685.0,484978794.0,496366058.0,508318102.0,520655398.0,532809933.0,NaN
4,Angola,AGO,"Population, total",SP.POP.TOTL,5231654.0,5301583.0,5354310.0,5408320.0,5464187.0,5521981.0,...,30234839.0,31297155.0,32375632.0,33451132.0,34532429.0,35635029.0,36749906.0,37885849.0,39040039.0,NaN


In [4]:
population.columns

Index(['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code',
       '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968',
       '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977',
       '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986',
       '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995',
       '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004',
       '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013',
       '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022',
       '2023', '2024', '2025', 'Unnamed: 70'],
      dtype='str')

In [5]:
population.shape

(266, 71)

In [6]:
metadata.head()

,Country Code,Region,IncomeGroup,SpecialNotes,TableName,Unnamed: 5
0,ABW,Latin America & Caribbean,High income,NaN,Aruba,NaN
1,AFE,NaN,NaN,"26 countries, stretching from the Red Sea in t...",Africa Eastern and Southern,NaN
2,AFG,Middle East & North Africa,Low income,The reporting period for national accounts dat...,Afghanistan,NaN
3,AFW,NaN,NaN,"22 countries, stretching from the westernmost ...",Africa Western and Central,NaN
4,AGO,Sub-Saharan Africa,Lower middle income,The World Bank systematically assesses the app...,Angola,NaN


In [7]:
metadata.shape

(265, 6)

## Description of the data

We loaded four files.

Indicator files (population, GDP per capita, internet users)
- Each row is one country or region. Every year from 1960 to 2025 has its own column, so you read across a row to see how a country changed over time.
- Each file has 266 rows and 71 columns (see `population.shape` above).
- Columns: `Country Name`, `Country Code` (a 3-letter code, e.g. USA), `Indicator Name`, `Indicator Code`, one column per year, and an empty column at the end caused by an extra comma in the file.

Metadata file
- Each row is one country or region, with the world region it belongs to (e.g. North America) and its income group (low, lower middle, upper middle, or high income).
- It has 265 rows (see `metadata.shape` above).

Things to be aware of
1. Each indicator file starts with 4 lines of notes before the real table, so we skip them when loading.
2. Years are spread across columns. To chart and combine the data, we reshape it so there is a single year column.
3. The files include 48 groups that are not countries, such as "World" and "High income". If we left them in, they would be counted as if they were countries. We can spot them because their region is blank in the metadata file.
4. Some values are missing. 2025 is mostly empty, so we use 2000-2024. **North Korea, one of the main attacker countries, has no GDP data and internet data for only 5 of the 25 years.**
5. The World Bank uses 3-letter country codes (USA) and official names (Russian Federation), while EuRepoC and Ransomware.live use 2-letter codes (US). We add a 2-letter code so the datasets can be matched.

## Data Cleaning

In [8]:
# helper: turn one indicator file from wide (one column per year) to long (one year column)
def to_long(data, value_name):
    long = data.melt(id_vars=['Country Name', 'Country Code'],
                     value_vars=[str(y) for y in range(2000, 2025)],
                     var_name='year', value_name=value_name)
    return long

# keep 2000-2024 (2025 is mostly empty) and melt each file
pop_long = to_long(population, 'population')
gdp_long = to_long(gdp, 'gdp_per_capita')
net_long = to_long(internet, 'internet_users_pct')

pop_long.head()

,Country Name,Country Code,year,population
0,Aruba,ABW,2000,90588.0
1,Africa Eastern and Southern,AFE,2000,406156661.0
2,Afghanistan,AFG,2000,20130327.0
3,Africa Western and Central,AFW,2000,274612131.0
4,Angola,AGO,2000,16194869.0


In [9]:
# combine the 3 indicators into one table: one row per country per year
wb = pop_long.merge(gdp_long, on=['Country Name', 'Country Code', 'year'])
wb = wb.merge(net_long, on=['Country Name', 'Country Code', 'year'])

# rename columns so they are easier to read
wb = wb.rename(columns={'Country Name': 'country', 'Country Code': 'iso3'})

wb.head()

,country,iso3,year,population,gdp_per_capita,internet_users_pct
0,Aruba,ABW,2000,90588.0,20681.023027,15.442823
1,Africa Eastern and Southern,AFE,2000,406156661.0,706.728441,NaN
2,Afghanistan,AFG,2000,20130327.0,174.930991,NaN
3,Africa Western and Central,AFW,2000,274612131.0,519.642600,NaN
4,Angola,AGO,2000,16194869.0,563.733796,0.105046


In [10]:
# keep only the columns we need from the metadata and rename them
meta = metadata[['Country Code', 'Region', 'IncomeGroup']]
meta = meta.rename(columns={'Country Code': 'iso3', 'Region': 'region', 'IncomeGroup': 'income_level'})

# remove regions like "World" and "High income" (they have no Region in the metadata)
real_countries = meta.dropna(subset=['region'])
print(len(meta), 'rows in metadata,', len(real_countries), 'real countries')

# keep only real countries and add region and income level
wb = wb.merge(real_countries, on='iso3', how='inner')
print(wb.shape)

265 rows in metadata, 217 real countries
(5425, 8)


In [11]:
# rename income levels to shorter labels for charts
wb['income_level'] = wb['income_level'].replace({'Low income': 'Low',
                                                 'Lower middle income': 'Lower middle',
                                                 'Upper middle income': 'Upper middle',
                                                 'High income': 'High'})

# shorten long country names so they read better in charts
wb['country'] = wb['country'].replace({'Russian Federation': 'Russia',
                                       'Iran, Islamic Rep.': 'Iran',
                                       "Korea, Dem. People's Rep.": 'North Korea',
                                       'Korea, Rep.': 'South Korea',
                                       'Egypt, Arab Rep.': 'Egypt',
                                       'Turkiye': 'Turkey',
                                       'Viet Nam': 'Vietnam'})

# make year a whole number
wb['year'] = wb['year'].astype(int)

In [12]:
# check missing values
wb.isna().sum()

country                 0
iso3                    0
year                    0
population              0
gdp_per_capita        183
internet_users_pct    535
region                  0
income_level            0
dtype: int64

In [13]:
# remove rows with no population (we can't compute "per million people" without it)
# we keep rows with missing GDP or internet use, so we don't lose the population data
wb = wb.dropna(subset=['population'])

# population in millions is easier to read and needed for "attacks per million people"
wb['population_millions'] = wb['population'] / 1000000

In [14]:
# add a 2-letter country code so we can join with EuRepoC and Ransomware.live
# (Channel Islands has no 2-letter code, so it shows as "not found")
codes = list(wb['iso3'].unique())
iso2_codes = coco.convert(codes, to='ISO2')
wb['iso2'] = wb['iso3'].map(dict(zip(codes, iso2_codes)))

print(wb.shape)
wb.head()

CHI not found in ISO3


(5425, 10)


,country,iso3,year,population,gdp_per_capita,internet_users_pct,region,income_level,population_millions,iso2
0,Aruba,ABW,2000,90588.0,20681.023027,15.442823,Latin America & Caribbean,High,0.090588,AW
1,Afghanistan,AFG,2000,20130327.0,174.930991,NaN,Middle East & North Africa,Low,20.130327,AF
2,Angola,AGO,2000,16194869.0,563.733796,0.105046,Sub-Saharan Africa,Lower middle,16.194869,AO
3,Albania,ALB,2000,3089027.0,1160.420471,0.114097,Europe & Central Asia,Upper middle,3.089027,AL
4,Andorra,AND,2000,65685.0,21810.248743,10.538836,Europe & Central Asia,High,0.065685,AD


What we did to clean the data: skipped the note rows, reshaped the years into one column, combined the three indicators into one table, removed the 48 non-country groups, added region and income level, shortened long labels (for example "Russian Federation" became "Russia"), removed rows with no population, and added population in millions and a 2-letter country code.

**After cleaning, each row is one country in one year.**

## EDA

In [15]:
# most recent year with good coverage
latest = wb[wb['year'] == 2024]

# top 10 countries by population
top_pop = latest.sort_values('population', ascending=False).head(10)

alt.Chart(top_pop).mark_bar().encode(
    alt.X('population_millions:Q', title='Population (millions)'),
    alt.Y('country:N', sort='-x', title='Country')
).properties(title='Most Populated Countries (2024)', width=450)

alt.Chart(...)

In [16]:
# internet use over time for the main countries in our cyber story
story = ['United States', 'China', 'Russia', 'Iran', 'Ukraine', 'Germany', 'United Kingdom', 'Israel']
story_df = wb[wb['country'].isin(story)]

alt.Chart(story_df).mark_line(point=True).encode(
    alt.X('year:O', title='Year'),
    alt.Y('internet_users_pct:Q', title='Internet users (% of population)', scale=alt.Scale(domain=[0, 100])),
    alt.Color('country:N', title='Country')
).properties(title='Internet Use in Key Cyber Attack Countries', width=500)

alt.Chart(...)

In [17]:
# how connected are countries around the world?
alt.Chart(latest).mark_bar().encode(
    alt.X('internet_users_pct:Q', bin=alt.Bin(step=10), title='Internet users (% of population)'),
    alt.Y('count():Q', title='Number of countries')
).properties(title='Internet Use Across Countries (2024)', width=450)

alt.Chart(...)

In [18]:
# number of countries in each income level
income_order = ['Low', 'Lower middle', 'Upper middle', 'High']

alt.Chart(latest).mark_bar().encode(
    alt.X('income_level:N', sort=income_order, title='Income level'),
    alt.Y('count():Q', title='Number of countries')
).properties(title='Countries by World Bank Income Level', width=350)

alt.Chart(...)

In [19]:
# average internet use by income level
by_income = latest.groupby('income_level')['internet_users_pct'].mean().reset_index()

alt.Chart(by_income).mark_bar().encode(
    alt.X('income_level:N', sort=income_order, title='Income level'),
    alt.Y('internet_users_pct:Q', title='Average internet users (% of population)', scale=alt.Scale(domain=[0, 100]))
).properties(title='Richer Countries Are More Connected (2024)', width=350)

alt.Chart(...)

In [20]:
# do richer countries have more internet users? (scatter plot for EDA only)
alt.Chart(latest).mark_circle(opacity=0.6, size=50).encode(
    alt.X('gdp_per_capita:Q', scale=alt.Scale(type='log'), title='GDP per capita (US$, log scale)'),
    alt.Y('internet_users_pct:Q', title='Internet users (% of population)'),
    alt.Color('income_level:N', sort=income_order, title='Income level')
).properties(title='GDP per Capita vs. Internet Use (2024)', width=450)

alt.Chart(...)

## Summary

What the data shows so far
- The cleaned table has 5,425 rows: 217 countries over 25 years (2000-2024).
- Countries differ hugely in size. In 2024, India had about 1,451 million people and China about 1,409 million, more than four times the United States (about 340 million). Simply counting attacks would make big countries look the most targeted, so we will also look at attacks per million people.
- Nearly every country in our cyber story went from barely online to almost fully online. Between 2000 and 2024, internet use rose from 1.8% to 92.0% in China, 2.0% to 94.4% in Russia, 0.9% to 85.3% in Iran, 0.7% to 82.5% in Ukraine, and 43.1% to 94.7% in the United States. This lines up with the rise in cyber incidents seen in the EuRepoC data after 2010.
- **Wealth and internet access go together.** In 2024, about 92% of people in high-income countries used the internet, compared with 79% in upper-middle, 57% in lower-middle, and just 23% in low-income countries.
- The world is divided: of the 182 countries with 2024 internet data, 99 have at least 80% of people online, while 38 have fewer than half.

Questions worth exploring
- Once we account for population, which countries are hit hardest per person? Small, wealthy, well-connected countries such as Israel or Estonia might rise to the top.
- Do richer countries have more ransomware victims per person? Ransomware gangs are after money, so they may focus on wealthy places.
- Does internet use or wealth better explain attacks? Since the main countries in our story are all above 80% online, income level may be more useful for telling them apart.

Does it need more data?
Yes. This dataset has no information about cyber attacks on its own. Its role is to add context to EuRepoC and Ransomware.live. All three can be matched using the 2-letter country code and the year (EuRepoC's `receiver_country_alpha_2_code`, Ransomware.live's `country`, and our new `iso2` column).

Visualizations we would like to make
1. A world map of cyber incidents per million people, shown next to a map of raw counts, to show how the picture changes once population is taken into account.
2. A bar chart of ransomware victims per million people for each income group, from low to high.
3. A ranked bar chart of the 15 countries with the most attacks per million people (only countries with at least 1 million people, so tiny countries don't dominate).
4. Small side-by-side line charts comparing internet growth with yearly cyber incidents for the key countries, instead of one chart with two y-axes.